### Structured output
Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic
Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"]=os.getenv("GROQ_API_KEY")
model=init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002829C0E7620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002829C2B0590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [2]:
from pydantic import BaseModel, Field


class Movie(BaseModel):
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The release year of the movie")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The rating of the movie out of 10")

In [9]:
model_with_structure = model.with_structured_output(Movie)

model_with_structure

RunnableBinding(bound=ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000002829C0E7620>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000002829C2B0590>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********')), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'The release year of the movie', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'rating': {'description': 'The rating of the movie out of 10', 'type': 'number'}}, 'required': ['title', 'year'

In [7]:
model.invoke("What is the movie Bahubalii about?")

AIMessage(content='<think>\nOkay, the user is asking about the movie "Bahubali." First, I need to confirm the correct title. They wrote "Bahubalii," but I think it\'s actually "Baahubali." Let me check that. Yes, the correct title is "Baahubali." It\'s a popular Indian film series.\n\nNow, the user wants to know what the movie is about. I should start by giving a brief overview. It\'s an action fantasy epic directed by S.S. Rajamouli, right? The series consists of two parts: "Baahubali: The Beginning" (2015) and "Baahubali: The Conclusion" (2017). Both are in Telugu but have been dubbed into multiple languages.\n\nThe story revolves around the Mahishmati kingdom. The main characters are Bahubali and his half-brother Bhallala Deva. There\'s a lot of political intrigue, power struggles, and a quest for justice. The user might not know the plot details, so I need to explain the setup. Bahubali is the son of King Amarendra Baahubali, who was deposed by his brother, Bhallala Deva. Bahubali 

In [10]:
response = model_with_structure.invoke("What is the movie Bahubali about?")
print(response)

title='Bahubali' year=2015 director='S.S. Rajamouli' rating=8.3


### Message output alongside parsed structure

In [11]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details."""
    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)  

response = model_with_structure.invoke("What is the movie Bahubali about?")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking about the movie Bahubali. Let me check if I have the necessary tool to provide details. The available tool is the Movie function, which requires title, year, director, and rating. I need to recall if I have that information stored.\n\nWait, Bahubali is a popular Indian film. The title is definitely "Bahubali". The year it was released was 2015. The director is S.S. Rajamouli. As for the rating, it\'s quite high; I think around 8.2 on IMDb. So, I can use the Movie function to provide this structured information. Let me make sure all required parameters are included: title, year, director, and rating. Yes, that\'s covered. I\'ll format the tool call with these details.\n', 'tool_calls': [{'id': 'a176sk7kx', 'function': {'arguments': '{"director":"S.S. Rajamouli","rating":8.2,"title":"Bahubali","year":2015}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completio

### Nested Structure

In [12]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    director: str
    rating: float
    actors: list[Actor]
    genres: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)
response = model_with_structure.invoke("Provide details about the movie Bahubali")
response

MovieDetails(title='Bahubali', year=2015, director='S. S. Rajamouli', rating=8.0, actors=[Actor(name='Prabhas', role='Bahubali'), Actor(name='Rana Daggubati', role='Bhallala Deva'), Actor(name='Tamannaah Bhatia', role='Devasena')], genres=['Action', 'Adventure', 'Drama'], budget=100.0)

### TypedDict
TypedDict provides a simpler alternative using Python’s built-in typing, ideal when you don’t need runtime validation.

In [13]:
from typing_extensions import Annotated, TypedDict

class MovieDict(TypedDict):
    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The release year of the movie"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The rating of the movie out of 10"]

model_with_structure = model.with_structured_output(MovieDict)
response = model_with_structure.invoke("Provide details about the movie Avengers: Endgame")
response

{'director': 'Anthony Russo, Joe Russo',
 'rating': 8.4,
 'title': 'Avengers: Endgame',
 'year': 2019}

In [15]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    director: str
    rating: float
    actors: list[Actor]
    genres: list[str]
    budget: float | None = Field(default=None, description="The budget of the movie in millions USD")
model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie Spiderman: No Way Home")
response

{'actors': [{'name': 'Tom Holland', 'role': 'Peter Parker'},
  {'name': 'Zendaya', 'role': 'MJ (Michelle Jones)'},
  {'name': 'Andrew Garfield', 'role': 'Spider-Man (Alternate Universe)'},
  {'name': 'Tobey Maguire', 'role': 'Spider-Man (Alternate Universe)'},
  {'name': 'Jared Leto', 'role': 'Doctor Octopus'}],
 'budget': 200000000,
 'director': 'Jon Watts',
 'genres': ['Action', 'Superhero', 'Family'],
 'rating': 8.5,
 'title': 'Spiderman: No Way Home',
 'year': 2021}

In [16]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### When to Use Which
* Use TypedDict when you want type-safe dictionaries for in-memory data structures and your primary goal is to use static analysis tools like mypy to catch errors during development.
* Use Pydantic when handling data from external sources like APIs or user inputs that need robust validation, error handling, and serialization (e.g., to JSON). It acts like a "bouncer" at the door, checking that your data meets the rules before letting it into your application.

### DataClasses
A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the @dataclass decorator

In [22]:
import os
from dotenv import load_dotenv
load_dotenv()
os.environ["GPT-4-MINI_API_KEY"]=os.getenv("GPT-4-MINI_API_KEY")

In [24]:
from pydantic import BaseModel, Field
from langchain.agents import create_agent

class ContactInfo(BaseModel):
    name: str
    email: str
    phone: str

llm = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("GPT-4-MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)


agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result

{'messages': [HumanMessage(content='Extract contact info from: John Doe, john@example.com, (555) 123-4567', additional_kwargs={}, response_metadata={}, id='9f3424ad-8bbf-4f9d-95b4-49a1d320daed'),
  AIMessage(content='{"name":"John Doe","email":"john@example.com","phone":"(555) 123-4567"}', additional_kwargs={'parsed': None, 'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 29, 'prompt_tokens': 84, 'total_tokens': 113, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4.1-mini-2025-04-14', 'system_fingerprint': 'fp_b6f445fc1c', 'id': 'chatcmpl-DLmADG1PavqQUJJhtR9qIiHME3ZO6', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--019d0f8d-13e2-7d91-991d-99eff0ef227c-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 84, 'output_tokens': 29, 

In [25]:
result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')

In [26]:
from typing_extensions import TypedDict
from langchain.agents import create_agent

class ContactInfo(TypedDict):
    name: str
    email: str
    phone: str

llm = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("GPT-4-MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)


agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}

In [27]:
from dataclasses import dataclass
from langchain.agents import create_agent

@dataclass
class ContactInfo:
    name: str
    email: str
    phone: str

llm = init_chat_model(
    model="gpt-4.1-mini",
    model_provider="openai",
    api_key=os.getenv("GPT-4-MINI_API_KEY"),
    base_url="https://models.inference.ai.azure.com"
)


agent = create_agent(
    model=llm,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [{"role": "user", "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"}]
})

result["structured_response"]

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')